<a href="https://colab.research.google.com/github/ManasDasri/NNDL/blob/main/notebooks/3_problem1_legal_bert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3 · Problem 1 — does legal pretraining help?

BERT against Legal-BERT. They share an architecture, a parameter count, a context
window and a chunk width, and differ only in pretraining corpus, so any gap between
them is attributable to domain adaptation.

**Needs a GPU.** Roughly 30–50 minutes for both on a T4.


## Setup

Clone the repository and install. The data layer needs nothing beyond the standard
library, so this is only for the model code.


In [1]:
!git clone -q https://github.com/ManasDasri/NNDL.git
%cd NNDL
!pip install -q -e . 'matplotlib>=3.8'

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — set Runtime > Change runtime type > T4 GPU')


/content/NNDL
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for legal-risk-clause-classifier (pyproject.toml) ... done
GPU: Tesla T4


### Prepare the corpus

Extracts answer spans from `CUAD_v1.json` and assigns document-level splits.
Chunking happens at training time, because each model needs a different window.


In [2]:
!legal-risk-prepare --cuad_json data/CUAD_v1.json --out_dir data/processed


510 documents -> data/processed/documents.jsonl
splits {'train': 358, 'val': 76, 'test': 76} -> data/processed/splits.json
  Cap on Liability                   662 spans across  275 documents
  Non-Compete                        251 spans across  119 documents
  License Grant                      772 spans across  255 documents
  Audit Rights                       639 spans across  214 documents
  Termination for Convenience        245 spans across  183 documents
  Insurance                          554 spans across  166 documents


### The baseline first

The comparison needs a floor to read against, and the CNN trains in a couple of
minutes on a GPU. Without it the table below has nothing to say about whether
110M pretrained parameters are earning their cost.


In [3]:
!legal-risk-train-cnn --epochs 12 --batch_size 32 --output_dir outputs/cnn


{'train': 13829, 'val': 2905, 'test': 3314}
vocabulary: 30,968 types
pos_weight: 27x, 68x, 22x, 35x, 63x, 53x
mixed precision: on (float16)
{"epoch": 1, "train_loss": 1.2681516694210417, "val_macro_f1": 0.42943690435352383, "val_macro_average_precision": 0.5669721380882469}
{"epoch": 2, "train_loss": 0.8073426887530653, "val_macro_f1": 0.46172375037585206, "val_macro_average_precision": 0.586374973805694}
{"epoch": 3, "train_loss": 0.6997919996585223, "val_macro_f1": 0.5262118170584211, "val_macro_average_precision": 0.6068251582730234}
{"epoch": 4, "train_loss": 0.5980129185332712, "val_macro_f1": 0.4747446416475553, "val_macro_average_precision": 0.6302537314791706}
{"epoch": 5, "train_loss": 0.5863444349506987, "val_macro_f1": 0.5574537966980031, "val_macro_average_precision": 0.6178860972312593}
{"epoch": 6, "train_loss": 0.46663932856301377, "val_macro_f1": 0.6022188133386303, "val_macro_average_precision": 0.6249641990256402}
{"epoch": 7, "train_loss": 0.3982061566505366, "val_ma

### Train both

Identical settings. The only difference is the checkpoint.


In [4]:
!legal-risk-train-transformer --model bert       --epochs 3 --batch_size 16 --lr 2e-5 --output_dir outputs/bert


bert-base-uncased  window=300w overlap=100w max_length=512t
{'train': 13829, 'val': 2905, 'test': 3314}
config.json: 100% 570/570 [00:00<00:00, 2.99MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 266kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 17.8MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 4.02MB/s]

model.safetensors: downloading bytes:  43% 191M/440M [00:01<00:00, 267MB/s, 14.9MB/s  ]
model.safetensors: downloading bytes:  90% 396M/440M [00:01<00:00, 324MB/s, 34.1MB/s  ]
model.safetensors: reconstructing file:  30% 134M/440M [00:01<00:04, 72.7MB/s, 6.49MB/s  ]
model.safetensors: downloading bytes: 100% 415M/415M [00:02<00:00, 157MB/s, 37.6MB/s  ]
model.safetensors: reconstructing file: 100% 440M/440M [00:02<00:00, 167MB/s, 40.1MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 4059.44it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------

In [5]:
!legal-risk-train-transformer --model legal-bert --epochs 3 --batch_size 16 --lr 2e-5 --output_dir outputs/legal_bert


nlpaueb/legal-bert-base-uncased  window=300w overlap=100w max_length=512t
{'train': 13829, 'val': 2905, 'test': 3314}
config.json: 100% 1.02k/1.02k [00:00<00:00, 3.43MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 298kB/s]
vocab.txt: 100% 222k/222k [00:00<00:00, 23.7MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 761kB/s]
[transformers] You passed `num_labels=6` which is incompatible to the `id2label` map of length `2`.

pytorch_model.bin: downloading bytes:  55% 241M/440M [00:01<00:00, 306MB/s, 19.5MB/s  ]
pytorch_model.bin: downloading bytes:  89% 391M/440M [00:02<00:00, 253MB/s, 33.9MB/s  ]
pytorch_model.bin: reconstructing file:  61% 268M/440M [00:02<00:01, 131MB/s, 6.44MB/s  ]
pytorch_model.bin: downloading bytes: 100% 415M/415M [00:02<00:00, 141MB/s, 36.7MB/s  ]
pytorch_model.bin: reconstructing file: 100% 440M/440M [00:02<00:00, 150MB/s, 39.4MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 25917.30it/s]
[transformers] BertForSequenceClassification LOAD REP

### Evaluate on the same held-out contracts


In [6]:
for run in ('outputs/bert', 'outputs/legal_bert'):
    !legal-risk-evaluate --run_dir {run} --split test


Loading weights: 100% 199/199 [00:00<00:00, 4682.09it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider

### Read the per-class columns, not the macro score

The hypothesis is specific: Legal-BERT should gain most on labels carried by legal
vocabulary, and least on labels carried by ordinary English such as Insurance.
A uniform gain across all six would mean something other than domain knowledge is
responsible.


In [7]:
import json, pathlib

# Only compare runs that actually exist, so a missing one narrows the table
# instead of crashing after an hour of training.
CANDIDATES = (('CNN', 'cnn'), ('BERT', 'bert'), ('Legal-BERT', 'legal_bert'), ('Longformer', 'longformer'))
runs = {}
for name, directory in CANDIDATES:
    path = pathlib.Path('outputs') / directory / 'metrics.json'
    if path.exists():
        runs[name] = json.load(open(path))
    else:
        print(f'skipping {name}: no {path}')

assert runs, 'no finished runs found'
labels = [r['label'] for r in next(iter(runs.values()))['test_at_tuned_thresholds']['per_class']]

print(f"\n{'label':32s}" + ''.join(f'{n:>13s}' for n in runs))
for i, label in enumerate(labels):
    f1 = {n: r['test_at_tuned_thresholds']['per_class'][i]['f1'] for n, r in runs.items()}
    row = f"{label:32s}" + ''.join(f'{f1[n]:13.3f}' for n in runs)
    if 'BERT' in f1 and 'Legal-BERT' in f1:
        row += f"   Δ {f1['Legal-BERT'] - f1['BERT']:+.3f}"
    print(row)

macro = {n: r['test_at_tuned_thresholds']['macro_f1'] for n, r in runs.items()}
row = f"\n{'macro F1':32s}" + ''.join(f'{macro[n]:13.3f}' for n in runs)
if 'BERT' in macro and 'Legal-BERT' in macro:
    row += f"   Δ {macro['Legal-BERT'] - macro['BERT']:+.3f}"
print(row)


skipping Longformer: no outputs/longformer/metrics.json

label                                     CNN         BERT   Legal-BERT
Cap on Liability                        0.780        0.780        0.766   Δ -0.015
Non-Compete                             0.317        0.560        0.509   Δ -0.051
License Grant                           0.760        0.762        0.752   Δ -0.010
Audit Rights                            0.696        0.738        0.725   Δ -0.013
Termination for Convenience             0.544        0.621        0.685   Δ +0.064
Insurance                               0.841        0.876        0.871   Δ -0.004

macro F1                                0.656        0.723        0.718   Δ -0.005


### Before claiming a result

Run-to-run variance on identical settings is about **±0.03 macro F1**, from float
nondeterminism on GPU. A gap smaller than that is noise.

If the difference is under 0.03, rerun both across several seeds and report the
spread. The honest finding may be *"domain pretraining did not measurably help on
this task at this scale"* — which is a real result, and more interesting than a
margin inside the noise floor.


In [ ]:
# Several seeds, if the headline gap is small.
for seed in (1, 2, 3):
    for model, out in (('bert', 'bert'), ('legal-bert', 'legal_bert')):
        !legal-risk-train-transformer --model {model} --epochs 3 --seed {seed} --output_dir outputs/{out}_s{seed}


bert-base-uncased  window=300w overlap=100w max_length=512t
{'train': 13829, 'val': 2905, 'test': 3314}
Loading weights: 100% 199/199 [00:00<00:00, 4686.87it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect iden